# Audit Log — Bronze to Silver

This notebook processes the raw auction audit log from the **bronze** layer (`company_db.bronze.auditlog`) and delivers clean, deduplicated records into the **silver** layer (`company_db.silver.auditlog`).

**What it does:**
1. Filters the audit log to `Online Bid Received` activity records only.
2. Parses the compound `bid_price_user_id` field to extract `bidder_id` and `user_id` as integers.
3. Deduplicates records so each bidder (`user_id`) appears only once per run.
4. Merges the cleaned records into the silver Delta table — updating existing rows on `user_id` + `item_id` match, inserting new ones otherwise.

**Source:** `company_db.bronze.auditlog`  
**Target:** `company_db.silver.auditlog` @ `abfss://auction@company.dfs.core.windows.net/silver/auditlog`

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import PySpark SQL functions module — aliased as F for convenience throughout this notebook.
from pyspark.sql import functions as F

In [ ]:
# Read the raw auction summary dataset from the bronze layer.
# This is the source DataFrame that will be standardized and merged into the silver table.
audit_log = spark.table(f'{catalog}.bronze.auditlog').filter(F.col('batch_id') == batch_id)

In [ ]:
from pyspark.sql.functions import split

# Keep only 'Online Bid Received' rows — other activity types are not relevant for bidder analysis.
audit_log = audit_log.filter(audit_log.activity =='Online Bid Received')

# The bid_price_user_id field is a space-delimited compound string.
# Split it into an array so individual elements (bid price, user_id, bidder_id) can be extracted.
audit_log = audit_log.withColumn('bid_price_user_id', split(audit_log.bid_price_user_id, ' '))

In [ ]:
# Each element in the split array is a key=value pair (e.g. "bidder_id=123").
# Index [2] holds the bidder_id pair — split on '=' and take the value after the equals sign.
audit_log = audit_log.select("*", split(audit_log.bid_price_user_id[2], "=")[1].alias('bidder_id'))

# Index [1] holds the user_id pair — same pattern.
audit_log = audit_log.select("*", split(audit_log.bid_price_user_id[1], "=")[1].alias('user_id'))

In [ ]:
from pyspark.sql.types import IntegerType

# The extracted user_id and bidder_id values are strings after the split.
# Cast both to IntegerType so they can be used as numeric keys in joins and the MERGE.
audit_log = audit_log.withColumn('user_id', audit_log.user_id.cast(IntegerType())).withColumn('bidder_id', audit_log.bidder_id.cast(IntegerType()))

In [ ]:
# Deduplicate so each unique user_id appears only once —
# retains the first occurrence when multiple bids exist for the same bidder.
audit_log = audit_log.dropDuplicates(['user_id'])

# Drop the from_bidder_to_clerk column — not needed in the silver layer.
audit_log = audit_log.drop('from_bidder_to_clerk')

## Extracting Unique Bidder & User IDs

Each row in the audit log contains a compound `bid_price_user_id` field that encodes both the **bidder ID** and the **user ID**. This section:

1. Splits the compound field and parses out `bidder_id` and `user_id` as integers.
2. Deduplicates on `user_id` so each bidder appears only once.
3. Merges the cleaned records into the silver `auditlog` table — updating existing rows and inserting new ones.

In [ ]:

# Create (or fully replace) the silver auditlog Delta table at the ADLS location.
# Schema is defined explicitly so column types are enforced on every merge.

spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.auditlog 
(
time_of_bid_placed TIMESTAMP,
item_id INTEGER,
lot_number INTEGER,
activity STRING,
bid_price_user_id STRING,
path STRING,
ingestion_timestamp TIMESTAMP,
bidder_id INTEGER,
user_id INTEGER,
created_at TIMESTAMP,
updated_at TIMESTAMP,
batch_id STRING
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/auditlog';""")

In [ ]:
%sql
-- drop table if exists company_db.silver.auditlog

In [ ]:
# Register the transformed DataFrame as a temporary SQL view.
# The downstream MERGE statement references this view as its source.
audit_log.createOrReplaceTempView('audit_log_view')

In [ ]:
# Upsert cleaned audit log records into the silver table.
# Match on user_id + item_id: update all fields (and set updated_at) for existing rows,
# insert new rows with created_at and updated_at set to the current timestamp.

spark.sql(
f"""
MERGE INTO {catalog}.silver.auditlog as target
USING audit_log_view as source
ON target.user_id = source.user_id
AND
target.item_id = source.item_id
WHEN MATCHED 
AND target.batch_id >= source.batch_id 
THEN
   UPDATE SET 
   target.time_of_bid_placed = source.time_of_bid_placed,
   target.item_id = source.item_id,
   target.lot_number = source.lot_number,
   target.activity = source.activity,
   target.bid_price_user_id = source.bid_price_user_id,
   target.path = source.path,
   target.ingestion_timestamp = source.ingestion_timestamp,
   target.bidder_id = source.bidder_id,
   target.user_id = source.user_id,
   target.updated_at = current_timestamp(),
   target.batch_id = source.batch_id
WHEN NOT MATCHED THEN
   INSERT (time_of_bid_placed, item_id, lot_number, activity, bid_price_user_id, path, ingestion_timestamp, bidder_id, user_id, created_at, updated_at, batch_id)
   VALUES (source.time_of_bid_placed, source.item_id, source.lot_number, source.activity, source.bid_price_user_id, source.path, source.ingestion_timestamp, source.bidder_id, source.user_id, current_timestamp(), current_timestamp(), source.batch_id);
 """)